# CLOY Processing

This notebook is for you to work with the CLOY data folder.


**Author**: Serena Day   
**Created on:** Sep 29, 2026

**Table of Contents**

1. [Creating a Similarity DataFrame](#sec1)
2. [Calculating the Similarity Score](#sec2)
3. [Similarity Function](#sec3)
4. [Stop Words](#sec4)

<h2 id="sec1"> Creating a Similarity DataFrame </h2>

In [ ]:
# Testing reading from the jsonl
# Reading each "title" of the posts
import pandas as pd
from textblob import TextBlob
import json

path = r"C:\Users\syld9\OneDrive\Documents\Courses\CS 315\CS-315\week4-tasks\data\cs315-week4-data\RedditExplorer\cloy_chronological.jsonl"

with open(path, "r", encoding="utf-8") as f:
    counter = 0
    for line in f:
        data = json.loads(line.strip())
        title = data.get("title")
        print(title)
        counter += 1
    print(counter, " total posts")


In [ ]:
# Turning the title into a separate words using textblob
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        title = data.get("title")
        title_blob = TextBlob(title)
        print(title_blob.words)

In [ ]:
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        comments = data.get("comments")
        for i in range(len(comments)):
            comment_body = comments[i].get("comment_body")
            comment_blob = TextBlob(comment_body)
            print(comment_blob.words)
        #comments_blob = TextBlob(title)
        #print(title_blob.words)

In [32]:
# Creating a large nested list of all posts and every word in their title + comments
# Use TextBlob.word_counts

with open(path, "r", encoding="utf-8") as f:
    words = []
    for line in f:
        data = json.loads(line.strip())
        # getting the words from the title of the post
        title = data.get("title")
        title_blob = TextBlob(title)
        words_per_post = title_blob.words

        # getting the words from all the comments in the post
        comments = data.get("comments")
        for i in range(len(comments)):
            comment_body = comments[i].get("comment_body")
            comment_blob = TextBlob(comment_body)
            comment_words = comment_blob.words
            
            words_per_post = words_per_post + comment_words
        words.append(words_per_post)
    print(len(words))

3075


In [ ]:
# Confirming the code counted all the words in the posts
for i in range(len(words)):
    print(str(len(words[i])) + " words in post #" + str(i))

In [40]:
# Making the above code into a function:
def getTotalWordsPerPost(path):
    with open(path, "r", encoding="utf-8") as f:
        words = []
        for line in f:
            data = json.loads(line.strip())
            # getting the words from the title of the post
            title = data.get("title")
            title_blob = TextBlob(title)
            words_per_post = title_blob.words

            # getting the words from all the comments in the post
            comments = data.get("comments")
            for i in range(len(comments)):
                comment_body = comments[i].get("comment_body")
                comment_blob = TextBlob(comment_body)
                comment_words = comment_blob.words
                
                words_per_post = words_per_post + comment_words
            words.append(words_per_post)
        return words

In [41]:
# Creating a list of all words in all posts (without repeating any)
totalWords = []
for post in words:
    for word in post:
        if (word not in totalWords):
            totalWords.append(word)
print(len(totalWords))

28999


In [ ]:
# Creating a matrix of word occurrence per post

occurrenceMatrix = []

wordList = getTotalWordsPerPost(path)
for post in wordList:
    wordOccurrenceByPost = []
    for word in totalWords: 
        occurrence = post.count(word)
        wordOccurrenceByPost.append(occurrence)
    occurrenceMatrix.append(wordOccurrenceByPost)


In [ ]:
# Making the matrix into a pandas df and changing the column names to the total words
occurrence_df = pd.DataFrame(occurrenceMatrix)
occurrence_df.columns = totalWords

In [75]:
occurrence_df.head()

,Crash,Landing,On,You,has,been,created,ok,let,’,...,ora,scumbag,half-brothers,Eundong,🩵,distribution,episdode,charecters,yeh,🥰💕😍😘
0,1,1,1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,2,1,13,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,3,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,1,0,0,0,9,...,0,0,0,0,0,0,0,0,0,0


<h2 id="sec2"> Calculating the Similarity Score </h2>

In [64]:
import numpy as np
from numpy.linalg import norm

In [65]:
def cosineSimilarity(vec1,vec2):
    """Calculate the cosine similarity between two vectors"""
    V1 = np.array(vec1)
    V2 = np.array(vec2)
    cosine = np.dot(V1,V2)/(norm(V1)*norm(V2))
    return cosine

In [109]:
# getting every combination of posts 
import itertools

vector = occurrence_df.index
combinations = itertools.combinations(vector,2)
    

In [110]:
# getting the similarity score of each combination
similarity_matrix = []
for combo in combinations:
    vec1 = occurrence_df.loc[combo[0]]
    vec2 = occurrence_df.loc[combo[1]]

    similarityScore = cosineSimilarity(vec1,vec2)
    similarity_row = [combo[0], combo[1], similarityScore]
    similarity_matrix.append(similarity_row)

In [111]:
# Creating and formatting the data frame
similarity_df = pd.DataFrame(similarity_matrix)
similarity_df = similarity_df.rename(columns = {0:"vec1", 1: "vec2", 2:"Cos Similarity"})

In [112]:
similarity_df.head()

,vec1,vec2,Cos Similarity
0,0,1,0.030521
1,0,2,0.019819
2,0,3,0.000000
3,0,4,0.026043
4,0,5,0.014778


In [113]:
len(similarity_df.index)

4726275

In [114]:
# Exporting to csv
similarity_df.to_csv('CLOY_similarity')

<h2 id="sec3"> Similarity Function </h2>

In [148]:
def findSimilarThreads(threadID):
    '''prints out the threadID and subject title of the thread for the top 5 most similar threads'''
    comparisons1 = similarity_df[similarity_df["vec1"] == threadID]
    comparisons2 = similarity_df[similarity_df["vec2"] == threadID]
    comparisons = pd.concat([comparisons1,comparisons2])

    comparisons = comparisons.sort_values(by="Cos Similarity", ascending=False)
    top5 = comparisons.head(5)
    dataList = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data=json.loads(line.strip())
            dataList.append(data)

        # printing original post title and id
        original_title = dataList[threadID].get("title")
        original_postID = dataList[threadID].get("post_id")
        print("Input post " + original_postID + ": " + original_title)
        print()

        print("Similar posts:")
        for id in top5["vec2"].values:
            title = dataList[id].get("title")
            postID = dataList[id].get("post_id")
            print(postID + ": " + title)




In [150]:
findSimilarThreads(320)

Input post hdppef: Can somebody explain Se-Ri’s reaction in episode 14, when RJH and YSR are discussing their scar and RJH steals a kiss.

Similar posts:
o8zvda: The scar scene
sljoa9: I hate how on episode 8, Seri is acting stupid
qw639y: Se-ri’s sad expression? (episode 14)
oa2ztj: RJH shows YSR his scars, then kisses her. How would you best describe YSR’s reaction to RJH’s kiss?
ij9o10: 📺 'Crash Landing On You' Episode 5 -- Discussion Thread (Re-Watch Version) 🍿


<h2 id="sec4"> Stop Words </h2>

In [152]:
# Getting list of stop words:
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [154]:
# Creating a list of all words in all posts (without repeating any), excluding stop words
totalWords = []
for post in words:
    for word in post:
        if (word not in totalWords) and (word not in stop_words):
            totalWords.append(word)
print(len(totalWords))

28851


In [155]:
# Creating a matrix of word occurrence per post

occurrenceMatrix = []

wordList = getTotalWordsPerPost(path)
for post in wordList:
    wordOccurrenceByPost = []
    for word in totalWords: 
        occurrence = post.count(word)
        wordOccurrenceByPost.append(occurrence)
    occurrenceMatrix.append(wordOccurrenceByPost)

In [156]:
# Making the matrix into a pandas df and changing the column names to the total words
occurrence_df = pd.DataFrame(occurrenceMatrix)
occurrence_df.columns = totalWords

In [157]:
# Getting every combination of indices
import itertools

vector = occurrence_df.index
combinations = itertools.combinations(vector,2)

In [158]:
# getting the similarity score of each combination
similarity_matrix = []
for combo in combinations:
    vec1 = occurrence_df.loc[combo[0]]
    vec2 = occurrence_df.loc[combo[1]]

    similarityScore = cosineSimilarity(vec1,vec2)
    similarity_row = [combo[0], combo[1], similarityScore]
    similarity_matrix.append(similarity_row)

In [159]:
# Exploring to CSV
similarity_df.to_csv('CLOY_similarity_noStopWords')

In [160]:
# Finding similar threads
findSimilarThreads(320)

Input post hdppef: Can somebody explain Se-Ri’s reaction in episode 14, when RJH and YSR are discussing their scar and RJH steals a kiss.

Similar posts:
o8zvda: The scar scene
sljoa9: I hate how on episode 8, Seri is acting stupid
qw639y: Se-ri’s sad expression? (episode 14)
oa2ztj: RJH shows YSR his scars, then kisses her. How would you best describe YSR’s reaction to RJH’s kiss?
ij9o10: 📺 'Crash Landing On You' Episode 5 -- Discussion Thread (Re-Watch Version) 🍿


Removing the Stop words did not impact the similar threads to a given thread. 